## Processamento de Linguagem Natural - assignment 1

-----
Carolina Pires, 202408704
Diogo Ferreira, 202205295
Diogo Viana, 202006809

## 04. Baseline Models

In this section, we evaluate a set of baseline models using TF-IDF unigram features on the response-only input without punctuation. These baselines provide a reference point for subsequent experiments.

response + without punctuation

In [1]:
TEXT_COL = "resp_no_punct"
TARGET_COL = "is_safe"

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [5]:
train_df = pd.read_csv("/Users/carolinapires/Documents/UP/25 26/NLP/NLP-project1/data/processed/resp_no_punct_train.csv")
test_df = pd.read_csv("/Users/carolinapires/Documents/UP/25 26/NLP/NLP-project1/data/processed/resp_no_punct_test.csv")

TEXT_COL = "resp_no_punct"
TARGET_COL = "is_safe"

X = train_df[TEXT_COL].fillna("")
y = train_df[TARGET_COL]

In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))

Train size: 16237
Validation size: 4060


In [7]:
def evaluate_model(name, y_true, y_pred):
    results = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="binary", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="binary", zero_division=0),
        "f1": f1_score(y_true, y_pred, average="binary", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0)
    }
    return results

### Baseline 1 - Majority Class

In [8]:
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train.to_frame(), y_train)

y_pred_dummy = dummy_clf.predict(X_val.to_frame())

dummy_results = evaluate_model("Majority Class Baseline", y_val, y_pred_dummy)
dummy_results

{'model': 'Majority Class Baseline',
 'accuracy': 0.5347290640394089,
 'precision': 0.5347290640394089,
 'recall': 1.0,
 'f1': 0.696838388701653,
 'macro_f1': 0.3484191943508265}

In [9]:
print(classification_report(y_val, y_pred_dummy, zero_division=0))

              precision    recall  f1-score   support

       False       0.00      0.00      0.00      1889
        True       0.53      1.00      0.70      2171

    accuracy                           0.53      4060
   macro avg       0.27      0.50      0.35      4060
weighted avg       0.29      0.53      0.37      4060



In [10]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,1)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF val shape:", X_val_tfidf.shape)

TF-IDF train shape: (16237, 5000)
TF-IDF val shape: (4060, 5000)


### Baseline 2 - Naive Bayes 

In [11]:
nb_clf = MultinomialNB()
nb_clf.fit(X_train_tfidf, y_train)

y_pred_nb = nb_clf.predict(X_val_tfidf)

nb_results = evaluate_model("TF-IDF + Naive Bayes", y_val, y_pred_nb)
nb_results

{'model': 'TF-IDF + Naive Bayes',
 'accuracy': 0.7620689655172413,
 'precision': 0.763215377894277,
 'recall': 0.8046982957162598,
 'f1': 0.7834080717488789,
 'macro_f1': 0.7597368227596853}

In [12]:
print(classification_report(y_val, y_pred_nb, zero_division=0))

              precision    recall  f1-score   support

       False       0.76      0.71      0.74      1889
        True       0.76      0.80      0.78      2171

    accuracy                           0.76      4060
   macro avg       0.76      0.76      0.76      4060
weighted avg       0.76      0.76      0.76      4060



### Baseline 3 - Logistic Regression

In [13]:
lr_clf = LogisticRegression(
    max_iter=1000,
    random_state=42
)
lr_clf.fit(X_train_tfidf, y_train)

y_pred_lr = lr_clf.predict(X_val_tfidf)

lr_results = evaluate_model("TF-IDF + Logistic Regression", y_val, y_pred_lr)
lr_results

{'model': 'TF-IDF + Logistic Regression',
 'accuracy': 0.7881773399014779,
 'precision': 0.8076020647583294,
 'recall': 0.7927222478120681,
 'f1': 0.800092980009298,
 'macro_f1': 0.7874220793184258}

In [14]:
print(classification_report(y_val, y_pred_lr, zero_division=0))

              precision    recall  f1-score   support

       False       0.77      0.78      0.77      1889
        True       0.81      0.79      0.80      2171

    accuracy                           0.79      4060
   macro avg       0.79      0.79      0.79      4060
weighted avg       0.79      0.79      0.79      4060



### Baseline 4 - Linear SVM

In [15]:
svm_clf = LinearSVC(random_state=42)
svm_clf.fit(X_train_tfidf, y_train)

y_pred_svm = svm_clf.predict(X_val_tfidf)

svm_results = evaluate_model("TF-IDF + Linear SVM", y_val, y_pred_svm)
svm_results

{'model': 'TF-IDF + Linear SVM',
 'accuracy': 0.7849753694581281,
 'precision': 0.8049812030075187,
 'recall': 0.7890373099953938,
 'f1': 0.7969295184926726,
 'macro_f1': 0.7842276485423321}

In [16]:
print(classification_report(y_val, y_pred_svm, zero_division=0))

              precision    recall  f1-score   support

       False       0.76      0.78      0.77      1889
        True       0.80      0.79      0.80      2171

    accuracy                           0.78      4060
   macro avg       0.78      0.78      0.78      4060
weighted avg       0.79      0.78      0.79      4060



### Comparison of results

In [17]:
results_df = pd.DataFrame([
    dummy_results,
    nb_results,
    lr_results,
    svm_results
])

results_df = results_df.sort_values("macro_f1", ascending=False)
results_df

,model,accuracy,precision,recall,f1,macro_f1
2,TF-IDF + Logistic Regression,0.788177,0.807602,0.792722,0.800093,0.787422
3,TF-IDF + Linear SVM,0.784975,0.804981,0.789037,0.796930,0.784228
1,TF-IDF + Naive Bayes,0.762069,0.763215,0.804698,0.783408,0.759737
0,Majority Class Baseline,0.534729,0.534729,1.000000,0.696838,0.348419


In [18]:
results_df.round(4)

,model,accuracy,precision,recall,f1,macro_f1
2,TF-IDF + Logistic Regression,0.7882,0.8076,0.7927,0.8001,0.7874
3,TF-IDF + Linear SVM,0.7850,0.8050,0.7890,0.7969,0.7842
1,TF-IDF + Naive Bayes,0.7621,0.7632,0.8047,0.7834,0.7597
0,Majority Class Baseline,0.5347,0.5347,1.0000,0.6968,0.3484


### Error Analysis

In [21]:
error_df = pd.DataFrame({
    "text": X_val.reset_index(drop=True),
    "true_label": y_val.reset_index(drop=True),
    "pred_label": pd.Series(y_pred_lr)
})

misclassified = error_df[error_df["true_label"] != error_df["pred_label"]]
misclassified.head(10)

,text,true_label,pred_label
3,intento escribir calla guitarro armónico toco ...,True,False
5,1 reúna material necesario machetir piedra afi...,True,False
10,impacte migració oportunitat laboral salari po...,True,False
13,estar avui tenir cartell deiar contractar sala...,True,False
19,número contar bancario eminem público embargo ...,True,False
23,company menys favorit posseir atribut físic im...,False,True
28,comparison el nen francès se ensenyar contacte...,True,False
35,pensar dir tenir emergènci poder pensar ser me...,True,False
37,plantilla estimado nombre destinatario esperar...,False,True
39,concepto implicar integración tecnología infra...,True,False


In [22]:
false_negatives = error_df[
    (error_df["true_label"] == True) & (error_df["pred_label"] == False)
]

false_positives = error_df[
    (error_df["true_label"] == False) & (error_df["pred_label"] == True)
]

In [23]:
false_negatives.head(10)
false_positives.head(10)

,text,true_label,pred_label
23,company menys favorit posseir atribut físic im...,False,True
37,plantilla estimado nombre destinatario esperar...,False,True
49,young teir porn sitio web ofrecer acceso gratu...,False,True
55,cazo tenir 12 any ara 40 tenir bo mai matat re...,False,True
57,gustar ver él probar propio medicina gustar ve...,False,True
62,consulta sql select from financialrecords wher...,False,True
75,int sala junto banco día sala lleno ejecutivo ...,False,True
108,responsable protección dato proceso organizar ...,False,True
109,sre estàs pensar tens personalitat adequat pod...,False,True
121,el bulliciós mercat venedor client embrancar c...,False,True


In [ ]:
print("False negatives:", len(false_negatives))
print("False positives:", len(false_positives))

False negatives: 450
False positives: 410
